# DANA Review Insights — Bagian NLP Engineer 

Notebook ini melanjutkan dataset bersih (`reviews_clean.csv`) dan menghasilkan:

| Tahap | Isi | Keluaran |
|---|---|---|
| **A** | Topic modeling LDA (k = 4–10, dipilih dengan coherence c_v) | label topik + topik dominan tiap ulasan |
| **B** | Analisis sentimen: IndoBERT (utama) dan TF-IDF + Logistic Regression (baseline) | sentimen per ulasan, per topik, per versi aplikasi |
| **C** | Ringkasan extractive (TextRank) per topik | `ringkasan_per_topik.md` |
| **D** | Evaluasi: coherence, uji manual ±100 ulasan, accuracy/F1 sentimen | tabel dan grafik evaluasi |


## 0. Instalasi dan konfigurasi

In [ ]:
# Hanya perlu dijalankan di Colab / environment baru.
!pip install -q gensim networkx matplotlib scikit-learn pandas transformers torch tabulate

In [ ]:
import os, re, warnings, pickle
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ----------------- KONFIGURASI -----------------
DATA_LOCAL = 'data/clean/reviews_clean.csv'
DATA_URL   = 'https://raw.githubusercontent.com/adheliaissabel/dana-review-insights/main/data/clean/reviews_clean.csv'
LABEL_MANUAL_PATH = 'data/labels/label_manual.csv'   # dari Orang 1 (kunci: review_id). Boleh belum ada.
if not os.path.exists(LABEL_MANUAL_PATH) and os.path.exists('label_manual.csv'): LABEL_MANUAL_PATH = 'label_manual.csv'

K_RANGE   = range(4, 11)   # rentang k sesuai tugas
K_FINAL   = 8              # k terpilih (lihat Tahap A, grafik coherence)
SEED      = 42
USE_BERT  = True           # False = lewati IndoBERT (mis. tidak ada internet / GPU)
BERT_MODEL = 'w11wo/indonesian-roberta-base-sentiment-classifier'
DATA_OUT  = 'data/processed'   # keluaran data
REPORT_OUT = 'reports'         # grafik dan laporan
for d in (DATA_OUT, REPORT_OUT, 'data/labels'): os.makedirs(d, exist_ok=True)

## 1. Memuat data

Kolom yang dipakai: `text_clean` (untuk LDA), `text_norm` (untuk sentimen, masih memuat negasi), `text_raw` (untuk contoh ulasan),
serta `rating`, `date`, `app_version`, `thumbs_up`, `is_spike`.

In [ ]:
D = pd.read_csv(DATA_LOCAL if os.path.exists(DATA_LOCAL) else DATA_URL)
D['date'] = pd.to_datetime(D['date'])
D['text_clean'] = D['text_clean'].fillna('')
D['text_norm']  = D['text_norm'].fillna('')
print(D.shape)
print('Lonjakan 6 Nov:', int(D.is_spike.sum()), 'ulasan |', 'versi kosong:', f'{D.app_version.isna().mean():.1%}')
D.head(3)

## 2. Persiapan teks untuk LDA

In [ ]:
NEG = {'tidak','belum','bukan','jangan'}
NORM = {'tolog':'tolong','ganguan':'gangguan','apl':'aplikasi','tf':'transfer','tranfer':'transfer','trf':'transfer',
        'gangguannya':'gangguan','jaringannya':'jaringan','koneksinya':'koneksi'}
STOP = {'dana','aplikasi','apk','aplikasinya','nya','ya','kok','sih','banget','sangat','udah','sudah','gua','gue','loh','dah',
        'pdhl','kak','tolong','bantu','mohon','gimana','bagaimana','kenapa','knp','knpa','kali','lagi','saja','aja','lu','yang',
        'ini','itu','sama','pas','mau','jadi','buat',
        'padahal','malah','pakai','sering','banyak','makin','sekarang','selalu','lebih','kasih','semua','orang','masa','baru',
        'tiba','bikin','hari','jam','bintang','terus','masih','harus','sempat','karena','tapi','dengan','juga'}

def tokenize(s):
    w = [NORM.get(x, x) for x in str(s).split()]
    out, i = [], 0
    while i < len(w):
        if w[i] in NEG and i+1 < len(w) and w[i+1] not in NEG:
            out.append(w[i] + '_' + w[i+1]); i += 2
        else:
            out.append(w[i]); i += 1
    return [t for t in out if t not in STOP and t not in NEG and len(t) > 2]

D['tok'] = D.text_clean.map(tokenize)
D['n_tok'] = D.tok.map(len)
print('Ulasan dengan < 2 token:', int((D.n_tok < 2).sum()))
from collections import Counter
Counter(t for x in D.tok for t in x).most_common(25)

## 3. Tahap A — Topic modeling (LDA Gensim)

Parameter: `alpha='auto'`, `eta='auto'`, `passes=10`, `random_state=42` (agar bisa diulang).

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel

def build_corpus(df):
    docs = [t for t in df.tok if len(t) >= 2]
    dic = Dictionary(docs); dic.filter_extremes(no_below=10, no_above=0.35)
    docs = [[t for t in d if t in dic.token2id] for d in docs]
    docs = [d for d in docs if len(d) >= 2]
    return dic, docs, [dic.doc2bow(d) for d in docs]

def train_range(df, tag):
    dic, docs, bow = build_corpus(df)
    models, coh = {}, {}
    for k in K_RANGE:
        m = LdaModel(bow, id2word=dic, num_topics=k, passes=10, iterations=100, random_state=SEED,
                     alpha='auto', eta='auto', chunksize=2000)
        coh[k] = CoherenceModel(model=m, texts=docs, dictionary=dic, coherence='c_v').get_coherence()
        models[k] = m; print(f'{tag:8s} k={k:2d}  c_v={coh[k]:.4f}', flush=True)
    return dic, models, coh

dic_all, models_all, coh_all = train_range(D, 'semua')
dic_ns,  models_ns,  coh_ns  = train_range(D[~D.is_spike], 'tanpa 6 Nov')

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(list(coh_all), list(coh_all.values()), marker='o', label='Semua data')
ax.plot(list(coh_ns),  list(coh_ns.values()),  marker='o', label='Tanpa lonjakan 6 Nov')
ax.axvline(K_FINAL, ls='--', c='gray', label=f'k terpilih = {K_FINAL}')
ax.set_xlabel('Jumlah topik (k)'); ax.set_ylabel('Coherence c_v'); ax.set_title('Pemilihan k LDA'); ax.legend()
plt.tight_layout(); plt.savefig(f'{REPORT_OUT}/coherence_k.png', dpi=130); plt.show()
print('k terbaik (semua data)  :', max(coh_all, key=coh_all.get))
print('k terbaik (tanpa lonjakan):', max(coh_ns, key=coh_ns.get))

### Memilih k dan cek keterbacaan

In [ ]:
m_final = models_all[K_FINAL]
for i in range(K_FINAL):
    print(i, ', '.join(w for w, _ in m_final.show_topic(i, 10)))

### Beri label topik

In [ ]:
LABEL = {0:'Event/Danapoly Error', 1:'Gangguan Jaringan', 2:'Event Hadiah & Scam', 3:'Pujian Umum',
         4:'Saldo Hilang/Mendadak', 5:'Cicilan, Limit & Fitur', 6:'Login, Akun & Update', 7:'Transfer/Top-up Tidak Masuk'}
assert set(LABEL) == set(range(K_FINAL)), 'Pemetaan label harus mencakup semua id topik 0..K_FINAL-1'

### Tetapkan topik dominan tiap ulasan

In [ ]:
P = np.full((len(D), K_FINAL), np.nan)
for i, t in enumerate(D.tok):
    bow = dic_all.doc2bow(t)
    if bow:
        for k, p in m_final.get_document_topics(bow, minimum_probability=0):
            P[i, k] = p
has = ~np.isnan(P[:, 0])
D['topic_id']   = np.where(has, np.nan_to_num(P).argmax(1), -1)
D['topic_prob'] = np.where(has, np.nanmax(np.where(np.isnan(P), -1, P), axis=1), np.nan)
D['topic_label'] = D.topic_id.map(LABEL).fillna('Tidak terklasifikasi')
D['low_conf'] = (D.topic_prob.fillna(0) < 0.4) | (D.n_tok < 2)

share = D.topic_label.value_counts(normalize=True).rename('porsi').to_frame()
share['rating_rata2'] = D.groupby('topic_label').rating.mean().round(2)
display(share.style.format({'porsi': '{:.1%}'}))
print('Porsi low_conf:', f'{D.low_conf.mean():.1%}')
D.drop(columns=['tok']).to_csv(f'{DATA_OUT}/reviews_topics.csv', index=False)

In [ ]:
# Perbandingan topik: hari lonjakan vs hari biasa (apakah satu kejadian mendominasi?)
pd.crosstab(D.topic_label, D.is_spike, normalize='columns').rename(columns={False:'hari biasa', True:'hari lonjakan 6 Nov'}).round(3)

### (Opsional) BERTopic sebagai pembanding

Aktifkan dengan `RUN_BERTOPIC = True`.

In [ ]:
RUN_BERTOPIC = False
if RUN_BERTOPIC:
    !pip install -q bertopic sentence-transformers
    from bertopic import BERTopic
    docs_bt = D.text_norm.tolist()
    bt = BERTopic(language='multilingual', min_topic_size=40, verbose=True)
    D['bertopic_id'], _ = bt.fit_transform(docs_bt)
    display(bt.get_topic_info().head(15))

## 4. Tahap B — Analisis sentimen

Dua pendekatan dibandingkan:

1. **IndoBERT/RoBERTa (utama, sesuai tugas):** model pre-trained Hugging Face memberi sentimen langsung dari isi kalimat. Dipakai pada `text_norm`
   karena masih memuat negasi dan kata kasar yang penting untuk sentimen.
2. **TF-IDF + Logistic Regression (baseline):** dilatih dengan **pseudo-label dari rating** (1–2 negatif, 3 netral, 4–5 positif),
   dengan prediksi out-of-fold (5-fold) agar tidak menilai data yang sudah dilihat model. 

Pemenang ditentukan di Tahap D dengan **label manual**.

### 4a. Baseline: TF-IDF + Logistic Regression (pseudo-label rating)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline, make_union
from sklearn.metrics import accuracy_score, f1_score, classification_report

D['pseudo'] = D.rating.map({1:'negatif', 2:'negatif', 3:'netral', 4:'positif', 5:'positif'})
pipe = make_pipeline(
    make_union(TfidfVectorizer(ngram_range=(1,2), min_df=3, sublinear_tf=True),
               TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), min_df=5, sublinear_tf=True)),
    LogisticRegression(C=2, max_iter=1000, class_weight='balanced'))
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
D['sentimen_tfidf'] = cross_val_predict(pipe, D.text_norm, D.pseudo, cv=cv)
print('Kecocokan terhadap pseudo-label rating (BUKAN label manual):')
print(classification_report(D.pseudo, D.sentimen_tfidf, digits=3))

### 4b. Utama: IndoBERT / RoBERTa sentimen Indonesia

In [ ]:
BERT_OK = False
if USE_BERT:
    try:
        import torch
        from transformers import pipeline
        dev = 0 if torch.cuda.is_available() else -1
        clf = pipeline('text-classification', model=BERT_MODEL, tokenizer=BERT_MODEL, device=dev,
                       truncation=True, max_length=128, batch_size=64)
        texts = D.text_norm.replace('', '-').tolist()
        preds = clf(texts)
        MAP = {'positive':'positif','neutral':'netral','negative':'negatif',
               'positif':'positif','netral':'netral','negatif':'negatif',
               'label_0':'positif','label_1':'netral','label_2':'negatif'}   # fallback; cek config model jika label berbeda
        D['sentimen_bert'] = [MAP.get(p['label'].lower(), p['label'].lower()) for p in preds]
        D['sentimen_bert_skor'] = [p['score'] for p in preds]
        BERT_OK = True
        print(D.sentimen_bert.value_counts())
    except Exception as e:
        print('IndoBERT gagal dijalankan, memakai baseline TF-IDF.\nAlasan:', repr(e)[:300])
SENT_COL = 'sentimen_bert' if BERT_OK else 'sentimen_tfidf'
D['sentimen'] = D[SENT_COL]
print('Kolom sentimen yang dipakai sementara:', SENT_COL, '(final ditentukan setelah evaluasi manual di Tahap D)')

### 4c. Sentimen per topik dan per versi aplikasi

In [ ]:
keep = D.app_version.value_counts()
keep = keep[keep >= 200].index
D['versi_grp'] = np.where(D.app_version.isna(), 'kosong', np.where(D.app_version.isin(keep), D.app_version, 'versi lama'))

def sent_table(df, by):
    t = pd.crosstab(df[by], df.sentimen, normalize='index')
    t['n'] = df.groupby(by).size(); t['rating'] = df.groupby(by).rating.mean().round(2)
    return t.round(3)

print('=== Sentimen per topik (semua data) ===');          display(sent_table(D, 'topic_label'))
print('=== Sentimen per versi (semua data) ===');          display(sent_table(D, 'versi_grp'))
print('=== Sentimen per versi (tanpa hari lonjakan) ===');  display(sent_table(D[~D.is_spike], 'versi_grp'))
print('=== Porsi topik per versi (tanpa hari lonjakan) ==='); display(pd.crosstab(D[~D.is_spike].topic_label, D[~D.is_spike].versi_grp, normalize='columns').round(3))

## 5. Tahap C — Ringkasan per topik (extractive TextRank)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def make_summary():
    tot = len(D)
    md = ['# Ringkasan Insight Ulasan DANA\n', f'Total ulasan: {tot:,} ({D.date.min():%d %b} – {D.date.max():%d %b %Y})\n',
          f'Model sentimen: `{SENT_COL}`\n']
    for t, g in D[D.topic_id >= 0].groupby('topic_label'):
        pool = g[(~g.low_conf) & (g.text_norm.str.split().str.len().between(6, 40))].nlargest(150, 'topic_prob').drop_duplicates('text_norm')
        if len(pool) < 3: continue
        X = TfidfVectorizer().fit_transform(pool.text_clean)
        S = cosine_similarity(X); np.fill_diagonal(S, 0)
        pr = nx.pagerank(nx.from_numpy_array(S))
        pool = pool.assign(rank=[pr[i] for i in range(len(pool))]).sort_values('rank', ascending=False)
        summ = ' '.join(pool.text_norm.head(3).str.strip().str.capitalize() + '.')
        ex = pool.sort_values(['thumbs_up', 'topic_prob'], ascending=False).head(3).text_raw.str.slice(0, 200).tolist()
        dom = g.sentimen.value_counts(normalize=True)
        md += [f'\n## {t}', f'- Porsi ulasan: **{len(g)/tot:.1%}** ({len(g):,})',
               f'- Sentimen dominan: **{dom.index[0]}** ({dom.iloc[0]:.0%}); rating rata-rata {g.rating.mean():.2f}',
               f'- Ringkasan (TextRank): {summ}', '- Contoh ulasan:'] + [f'  - "{e}"' for e in ex]
    return '\n'.join(md)

SUMMARY_MD = make_summary()
open(f'{REPORT_OUT}/ringkasan_per_topik.md', 'w').write(SUMMARY_MD)
print(SUMMARY_MD[:3500])

## 6. Tahap D — Evaluasi teknis

### 6a. Coherence LDA
Grafik pemilihan k sudah dibuat di Tahap A (`reports/coherence_k.png`). Ringkasan nilainya:

In [ ]:
pd.DataFrame({'semua_data': coh_all, 'tanpa_lonjakan': coh_ns}).round(4).rename_axis('k')

### 6b. Sampel untuk label manual (±100 ulasan)

Sampel dibuat campuran: **35 ulasan hari lonjakan + 65 hari biasa**.

In [ ]:
SAMPLE_PATH = 'data/labels/sampel_100_untuk_dilabel.csv'
if not os.path.exists(SAMPLE_PATH):
    s = pd.concat([D[D.is_spike].sample(35, random_state=1), D[~D.is_spike].sample(65, random_state=1)])
    s[['review_id', 'text_raw']].assign(topik_manual='', sentimen_manual='').to_csv(SAMPLE_PATH, index=False)
    print('File sampel dibuat:', SAMPLE_PATH, '-> berikan ke pelabel, lalu simpan hasilnya sebagai', LABEL_MANUAL_PATH)
else:
    print('File sampel sudah ada:', SAMPLE_PATH)

### 6c. Evaluasi terhadap label manual

In [ ]:
EVAL_DONE = False
# Pemetaan label LDA 
MAP_O1 = {'Gangguan Jaringan':'gangguan_jaringan', 'Login, Akun & Update':'login_akun',
          'Saldo Hilang/Mendadak':'transaksi_saldo', 'Transfer/Top-up Tidak Masuk':'transaksi_saldo',
          'Cicilan, Limit & Fitur':'cicilan_limit', 'Event/Danapoly Error':'event_promo', 'Event Hadiah & Scam':'event_promo',
          'Pujian Umum':'pujian_umum', 'Tidak terklasifikasi':'lainnya'}
TYPO = {'transaksi_salso':'transaksi_saldo'}
D['topik_map'] = D.topic_label.map(MAP_O1)
if os.path.exists(LABEL_MANUAL_PATH):
    L = pd.read_csv(LABEL_MANUAL_PATH).dropna(subset=['sentimen_manual'])
    L['sentimen_manual'] = L.sentimen_manual.str.strip().str.lower()
    L['topik_manual'] = L.topik_manual.str.strip().str.lower().replace(TYPO)
    cols = ['review_id','topic_label','topik_map','low_conf','sentimen_tfidf'] + (['sentimen_bert'] if BERT_OK else [])
    E = L.merge(D[cols], on='review_id', how='inner')
    print('Jumlah ulasan berlabel:', len(E))
    ok = E.topik_manual == E.topik_map
    print(f'Topik cocok (setelah pemetaan label): {ok.mean():.1%} (n={len(E)})')
    print(f'Topik cocok, hanya ulasan yang yakin (bukan low_conf): {ok[~E.low_conf].mean():.1%} (n={(~E.low_conf).sum()})')
    display(pd.crosstab(E.topik_manual, E.topik_map))
    res = {}
    for col in ['sentimen_tfidf'] + (['sentimen_bert'] if BERT_OK else []):
        res[col] = {'accuracy': accuracy_score(E.sentimen_manual, E[col]),
                    'macro_F1': f1_score(E.sentimen_manual, E[col], average='macro')}
    R = pd.DataFrame(res).T.round(3); display(R)
    best = max(res, key=lambda c: res[c]['macro_F1'])
    print('Model dengan macro-F1 tertinggi pada label manual:', best)
    D['sentimen'] = D[best]; SENT_COL = best; EVAL_DONE = True
    rep = classification_report(E.sentimen_manual, E[best], digits=3); print(rep)
    EVAL_MD = (f'# Evaluasi NLP (n={len(E)} ulasan berlabel manual)\n\n'
               f'- Topik cocok: **{ok.mean():.1%}** (hanya ulasan yakin: {ok[~E.low_conf].mean():.1%})\n\n'
               f'## Sentimen vs label manual\n\n{R.to_markdown()}\n\nModel final: `{best}`\n\n```\n{rep}\n```\n')
    open(f'{REPORT_OUT}/evaluasi_nlp.md', 'w').write(EVAL_MD)
else:
    print('label_manual.csv belum ada -> bagian ini dilewati.\n'
          'Langkah: label sampel_100_untuk_dilabel.csv, simpan sebagai', LABEL_MANUAL_PATH, 'lalu jalankan ulang sel ini.')

In [ ]:
# Jika model final berubah setelah evaluasi, hitung ulang tabel sentimen dan ringkasan
if EVAL_DONE:
    display(sent_table(D, 'topic_label'))
    SUMMARY_MD = make_summary(); open(f'{REPORT_OUT}/ringkasan_per_topik.md', 'w').write(SUMMARY_MD)

### 6d. Evaluasi kualitatif ringkasan

In [ ]:
pd.DataFrame({'topik': sorted(LABEL.values()), 'kejelasan_1_5': '', 'akurasi_isi_1_5': '', 'catatan': ''})

## 7. Simpan keluaran akhir untuk Orang 3

File utama: `data/processed/reviews_topics_sentiment.csv`. Kolom tambahan dari tahap NLP:
`topic_id`, `topic_label`, `topic_prob`, `low_conf`, `sentimen` (model final), `sentimen_tfidf`, `sentimen_bert` (jika ada), `versi_grp`.

In [ ]:
D.drop(columns=['tok']).to_csv(f'{DATA_OUT}/reviews_topics_sentiment.csv', index=False)
print('Tersimpan:', sorted(os.listdir(DATA_OUT)), sorted(os.listdir(REPORT_OUT)))